# Retrievers Deepdive


[Step 9 - Retrievers]

> **MLCourse - Agentic AI - Retrievers**
> Stage in the capstone: RETRIEVE - feeds the context block of the capstone RAG prompt.

# What you will learn

1. The retriever contract: query in, `Document` objects out - nothing else promised.
2. `vectorstore.as_retriever(search_kwargs={"k": 3})` and what `.invoke()` returns.
3. similarity vs MMR: watching overlapping Alice chunks get SUPPRESSED.
4. What `lambda_mult` does across its full 0-to-1 range.
5. Piping a retriever into LCEL and printing the assembled RAG prompt -
   completely offline, zero API keys.

The store (module 08) answers "what is indexed?"; the retriever answers "what
should I fetch for THIS question?" - and because it speaks the Runnable
protocol, it plugs straight into every chain you build from here on.

In [1]:
# ---------------------------------------------------------------------------
# Setup cell (identical in every MLCourse notebook): imports, TRACK walker,
# DATA folder creation, .env loading, matplotlib inline magic - guarded so
# the file also runs as a plain script outside Jupyter.
# ---------------------------------------------------------------------------
from pathlib import Path


def find_track(start: Path, target: str = "03_agentic_ai") -> Path:
    """Climb parent folders until a directory named ``target`` shows up."""
    for candidate in [start, *start.parents]:
        probe = candidate / target
        if probe.is_dir():
            return probe
    raise FileNotFoundError(
        f"Could not find '{target}' above {start}. "
        "Open this notebook from inside the MLCourse repository."
    )


TRACK = find_track(Path.cwd())       # .../MLCourse/03_agentic_ai
DATA = TRACK / "data"                # one shared data folder for the track
DATA.mkdir(exist_ok=True)            # no-op when it already exists

from dotenv import load_dotenv       # noqa: E402  reads KEY=value files

load_dotenv()                        # .env beside the current directory
load_dotenv(TRACK / ".env")          # .env at the track root

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass                             # magic only exists inside IPython/Jupyter

print("[setup] TRACK:", TRACK)
print("[setup] DATA :", DATA)

[setup] TRACK: D:\projects\python\MLCourse\03_agentic_ai
[setup] DATA : D:\projects\python\MLCourse\03_agentic_ai\data


### Fresh in-memory store from the shared corpus

Module 08 taught persistence; this module is about the RETRIEVAL interface,
so we rebuild a small ephemeral Chroma (no persist_directory = RAM only) to
keep runs deterministic. Same ingest recipe as module 08: chapter regex,
20k chars, 500/100 splitter - the overlap=100 setting deliberately creates
near-duplicate neighbours, which is exactly what MMR needs to show off.

In [2]:
import re
import numpy as np
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

ALICE_URL = "https://www.gutenberg.org/files/11/11-0.txt"
alice_path = DATA / "alice.txt"
if not alice_path.exists():                          # download once, reuse always
    import urllib.request
    urllib.request.urlretrieve(ALICE_URL, alice_path)

RAW = alice_path.read_text(encoding="utf-8-sig")[:20_000]

marks = list(re.finditer(r"^CHAPTER [IVX]+\.", RAW, flags=re.MULTILINE))
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

documents = []
for i, mark in enumerate(marks):
    seg_end = marks[i + 1].start() if i + 1 < len(marks) else len(RAW)
    for piece in splitter.split_text(RAW[mark.start():seg_end]):
        documents.append(Document(
            page_content=piece,
            metadata={"source": "alice", "chapter": str(i + 1), "chunk_id": len(documents)},
        ))

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2")

vectordb = Chroma.from_documents(          # NO persist_directory -> memory only
    documents,
    embedding=embeddings,
    collection_name="alice_retrievers",
)
print("[store] vectors:", len(vectordb.get()["ids"]), "| chunks tagged:", len(documents))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[store] vectors: 60 | chunks tagged: 60


### Store to retriever: one line, one contract

`as_retriever` wraps the store behind the uniform Runnable interface. Note
that search settings travel in `search_kwargs` - the retriever defers them to
the underlying store at call time rather than freezing copies.

In [3]:
chroma_retriever = vectordb.as_retriever(search_kwargs={"k": 3})

retrieved = chroma_retriever.invoke("What did Alice drink?")

print("invoke returned :", type(retrieved).__name__, "of",
      len(retrieved), "Documents")
for i, doc in enumerate(retrieved, start=1):
    snippet = doc.page_content[:64].replace("\n", " ")
    print(f"[{i}] ch.{doc.metadata['chapter']} id={doc.metadata['chunk_id']:>2}",
          "|", snippet, "...")

invoke returned : list of 3 Documents
[1] ch.1 id=25 | However, this bottle was _not_ marked “poison,” so Alice venture ...
[2] ch.1 id=23 | It was all very well to say “Drink me,” but the wise little Alic ...
[3] ch.1 id=22 | There seemed to be no use in waiting by the little door, so she  ...


### similarity vs MMR on genuinely repetitive text

chunk_overlap=100 means neighbouring chunks SHARE about 100 characters, so a
focused query often retrieves several near-copies that waste prompt space.
We quantify redundancy as the mean pairwise cosine between the k retrieved
chunks: close to 1.0 means "you fetched the same passage three times".
MMR fights this by penalizing similarity to ALREADY-PICKED documents.

In [4]:
QUERY = "down the rabbit hole"


def mean_pairwise_cosine(docs) -> float:
    """Mean cosine over all unordered pairs of retrieved docs (redundancy gauge)."""
    vecs = np.array(embeddings.embed_documents([d.page_content for d in docs]))
    unit = vecs / np.linalg.norm(vecs, axis=1, keepdims=True)
    sims = unit @ unit.T
    upper = sims[np.triu_indices(len(docs), k=1)]
    return float(upper.mean())


sim_docs = chroma_retriever.invoke(QUERY)

print("SIMILARITY k=3 (plain nearest neighbors):")
for i, doc in enumerate(sim_docs, start=1):
    snippet = doc.page_content[:60].replace("\n", " ")
    print(f"  [{i}] id={doc.metadata['chunk_id']:>2} | {snippet} ...")
print("  mean pairwise cosine: %.3f   <- high = redundant"
      % mean_pairwise_cosine(sim_docs))

mmr_retriever = vectordb.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 20},   # consider 20 candidates, keep 3
)
mmr_docs = mmr_retriever.invoke(QUERY)

print("\nMMR fetch_k=20, k=3 (redundancy penalized):")
for i, doc in enumerate(mmr_docs, start=1):
    snippet = doc.page_content[:60].replace("\n", " ")
    print(f"  [{i}] id={doc.metadata['chunk_id']:>2} | {snippet} ...")
print("  mean pairwise cosine: %.3f   <- lower = more diverse"
      % mean_pairwise_cosine(mmr_docs))

overlap = set(d.metadata["chunk_id"] for d in sim_docs) & \
          set(d.metadata["chunk_id"] for d in mmr_docs)
print("\nchunks kept by BOTH strategies:", sorted(overlap),
      "(MMR swaps redundant picks for informative ones)")

SIMILARITY k=3 (plain nearest neighbors):
  [1] id= 4 | In another moment down went Alice after it, never once consi ...
  [2] id= 3 | on, Alice started to her feet, for it flashed across her min ...
  [3] id= 0 | CHAPTER I. Down the Rabbit-Hole   Alice was beginning to get ...
  mean pairwise cosine: 0.625   <- high = redundant

MMR fetch_k=20, k=3 (redundancy penalized):
  [1] id= 4 | In another moment down went Alice after it, never once consi ...
  [2] id= 0 | CHAPTER I. Down the Rabbit-Hole   Alice was beginning to get ...
  [3] id=51 | As she said this she looked down at her hands, and was surpr ...
  mean pairwise cosine: 0.478   <- lower = more diverse

chunks kept by BOTH strategies: [0, 4] (MMR swaps redundant picks for informative ones)


### lambda_mult: the diversity dial

MMR scores each candidate as a blend:

    lambda_mult * relevance_to_query - (1 - lambda_mult) * redundancy_penalty

Sweep it end to end and watch behaviour flip: 1.0 collapses back toward pure
similarity ranking; 0.0 maximizes difference from already-picked chunks even
at the cost of raw relevance; 0.5 balances both.

In [5]:
sweep = {}
for lam in (0.0, 0.25, 0.5, 0.75, 1.0):
    dial = vectordb.as_retriever(
        search_type="mmr",
        search_kwargs={"k": 3, "fetch_k": 20, "lambda_mult": lam},
    )
    picked = dial.invoke(QUERY)
    sweep[lam] = (
        tuple(d.metadata["chunk_id"] for d in picked),
        mean_pairwise_cosine(picked),
    )

for lam, (ids, redundancy) in sweep.items():
    print("lambda_mult=%.2f -> chunks %s | mean pairwise cos %.3f"
          % (lam, ids, redundancy))
print("\n1.0 behaves like plain similarity; 0.0 chases maximum diversity.")

lambda_mult=0.00 -> chunks (4, 51, 49) | mean pairwise cos 0.441
lambda_mult=0.25 -> chunks (4, 51, 49) | mean pairwise cos 0.441
lambda_mult=0.50 -> chunks (4, 0, 51) | mean pairwise cos 0.478
lambda_mult=0.75 -> chunks (4, 3, 0) | mean pairwise cos 0.625
lambda_mult=1.00 -> chunks (4, 3, 0) | mean pairwise cos 0.625

1.0 behaves like plain similarity; 0.0 chases maximum diversity.


### Retriever inside LCEL - no LLM required

The capstone's RAG chain looks like:

    {"context": retriever, "question": passthrough} | prompt | model | parser

We keep everything EXCEPT the model: the dict step fans out (retriever feeds
context while the raw question passes through), PromptTemplate renders both,
and a stub "parser" prints the final prompt instead of calling any model.
If the printed prompt contains real retrieved passages, the whole plumbing
works - model choice becomes a swappable last step (module 10).

In [6]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

RAG_PROMPT = PromptTemplate.from_template(
    "You are a careful assistant.\n"
    "Answer ONLY using the context below.\n\n"
    "### Context\n{context}\n\n"
    "### Question\n{question}\n\n"
    "### Answer\n"
)


def format_docs(docs) -> str:
    """Flatten retrieved Documents into one bullet-list context block."""
    return "\n\n".join("- " + d.page_content.replace("\n", " ") for d in docs)


def llm_stub(prompt_value) -> str:
    """StrOutputParser-ish stand-in: SHOWS what would be sent, sends nothing."""
    bar = "=" * 64
    rendered = prompt_value.to_string()
    return (bar + "\n[PROMPT THAT WOULD GO TO AN LLM]\n" + bar + "\n"
            + rendered + bar + "\n[stub stops here - no model was called]")


rag_chain = (
    {"context": chroma_retriever | RunnableLambda(format_docs),  # branch A
     "question": RunnablePassthrough()}                          # branch B
    | RAG_PROMPT                                                 # fill template
    | RunnableLambda(llm_stub)                                   # fake last step
)

final_output = rag_chain.invoke("What did Alice find and drink?")
print(final_output[:1400])
print("[ok] retrieved context injected OFFLINE - zero API keys involved.")

[PROMPT THAT WOULD GO TO AN LLM]
You are a careful assistant.
Answer ONLY using the context below.

### Context
- It was all very well to say “Drink me,” but the wise little Alice was not going to do _that_ in a hurry. “No, I’ll look first,” she said, “and see whether it’s marked ‘_poison_’ or not”; for she had read several nice little histories about children who had got burnt, and eaten up by wild beasts and other unpleasant things, all because they _would_ not remember the simple rules their friends had taught them: such as, that a red-hot poker will burn you if you hold it too long;

- However, this bottle was _not_ marked “poison,” so Alice ventured to taste it, and finding it very nice, (it had, in fact, a sort of mixed flavour of cherry-tart, custard, pine-apple, roast turkey, toffee, and hot buttered toast,) she very soon finished it off.  *      *      *      *      *      *      *      *      *      *      *      *      *  *      *      *      *      *      *      *   “What a

### Takeaway

- `as_retriever` turns any store into a Runnable returning Documents.
- Overlapping chunks make plain similarity redundant; MMR with
  `fetch_k >> k` buys diversity, and `lambda_mult` tunes the trade-off.
- Retriever + fan-out dict + PromptTemplate + stub = a testable RAG skeleton
  you can validate BEFORE ever paying for a model call.

### Summary

We wrapped Chroma as a retriever, inspected invoke output, proved MMR
suppresses overlap-created duplicates with a pairwise-cosine gauge, swept
lambda_mult across its range, and piped the retriever into an LCEL chain that
printed a fully context-loaded prompt offline. Module 10 attaches the model
and completes your first true RAG loop.